# DS-2002 Final Project
## AdventureWorks Data Lakehouse (Medallion Architecture)
### PySpark + MySQL + MongoDB + Structured Streaming

### 1.0. Import Required Libraries

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

C:\Program Files\spark-3.5.4-bin-hadoop3


### 2.0. Instantiate Global Variables

In [17]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "adventureworks",
    "conn_props" : {
        "user" : "root",
        "password" : "#Hi10172004",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location" : "local",
    "user_name" : "",
    "password" : "",
    "cluster_name" : "",
    "cluster_subnet" : "",
    "db_name" : "adventureworks",
    "collection" : "",
    "null_column_threshold" : 0.5
}

# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'proj-data')

batch_dir = os.path.join(base_dir, 'batch')
stream_dir = os.path.join(base_dir, 'streaming')

sales_orders_stream_dir = os.path.join(stream_dir, 'sales_orders')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

sales_orders_output_bronze = os.path.join(database_dir, 'fact_sales_orders', 'bronze')
sales_orders_output_silver = os.path.join(database_dir, 'fact_sales_orders', 'silver')
sales_orders_output_gold   = os.path.join(database_dir, 'fact_sales_orders', 'gold')

### 3.0. Define Global Functions

In [18]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))
    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name', 'size', 'modification_time']
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
    print(f"The stream has processed {len(query.recentProgress)} batch(es)")


def remove_directory_tree(path: str):
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
    except Exception as e:
        return f"An error occurred: {e}"


def drop_null_columns(df, threshold):
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold]
    df_dropped = df.drop(*columns_with_nulls)
    return df_dropped


def get_mysql_dataframe(spark_session, sql_query: str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    dframe = spark_session.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("driver", args['conn_props']['driver']) \
        .option("user", args['conn_props']['user']) \
        .option("password", args['conn_props']['password']) \
        .option("query", sql_query) \
        .load()
    return dframe


def get_mongo_uri(**args):
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"
    return uri


def get_mongo_client(**args):
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())
    else:
        client = pymongo.MongoClient(mongo_uri)
    return client


def set_mongo_collections(mongo_client, db_name: str, data_directory: str, json_files: dict):
    db = mongo_client[db_name]
    for collection_name, filename in json_files.items():
        db.drop_collection(collection_name)
        json_file = os.path.join(data_directory, filename)
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
        collection = db[collection_name]
        collection.insert_many(json_object)
    mongo_client.close()


def get_mongodb_dataframe(spark_session, **args):
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()
    dframe = dframe.drop('_id')
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    return dframe

### 4.0. Initialize Data Lakehouse Directory Structure
Remove the Data Lakehouse Database Directory to Ensure Idempotency

In [19]:
print(remove_directory_tree(database_dir))

Directory 'C:\Users\aadit\Downloads\dssystems\DS-2002\FinalProj\spark-warehouse\adventureworks_dlh.db' has been removed successfully.


### 5.0. Create a New Spark Session

In [20]:
mysql_spark_jar = os.path.join(os.getcwd(), 'mysql-connector-j-9.1.0', 'mysql-connector-j-9.1.0.jar')

mongo_uri = get_mongo_uri(**mongodb_args)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DS-2002 Final Project – AdventureWorks Data Lakehouse")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.jars", mysql_spark_jar)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1")
    .config("spark.mongodb.input.uri", mongo_uri)
    .config("spark.mongodb.output.uri", mongo_uri)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", str(int(os.cpu_count())))
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
    .config("spark.sql.streaming.schemaInference", "true")
    .config("spark.sql.warehouse.dir", sql_warehouse_dir)
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("OFF")
spark

### 6.0. Create the Metadata Database

In [21]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Final Project – AdventureWorks Data Lakehouse'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Final Project');
"""
spark.sql(sql_create_db)

DataFrame[]

## Section II: Populate Dimensions by Ingesting 'Cold-path' Reference Data
### 1.0. Fetch Data from the File System
#### 1.1. Verify the location of the source data files on the file system

In [ ]:
get_file_info(batch_dir)

#### 1.2. Populate the **Employee Dimension**
##### 1.2.1. Read Data from CSV File

In [ ]:
employee_csv = os.path.join(batch_dir, 'dim_employee.csv')
print(employee_csv)

df_dim_employees = spark.read.format('csv').options(header='true', inferSchema='true').load(employee_csv)
df_dim_employees.toPandas().head(2)

##### 1.2.2. Make Necessary Transformations

In [ ]:
# Rename 'id' to 'employee_id'
df_dim_employees = df_dim_employees.withColumnRenamed("id", "employee_id")

# Add surrogate primary key via ROW_NUMBER()
df_dim_employees.createOrReplaceTempView("employees")
sql_employees = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY employee_id) AS employee_key
    FROM employees;
"""
df_dim_employees = spark.sql(sql_employees)

# Reorder columns – adjust list to match the actual columns in your CSV
ordered_columns = ['employee_key', 'employee_id', 'first_name', 'last_name',
                   'job_title', 'department', 'hire_date', 'email']

# Keep only columns that actually exist in the dataframe
existing_columns = [c for c in ordered_columns if c in df_dim_employees.columns]
df_dim_employees = df_dim_employees[existing_columns]
df_dim_employees.toPandas().head(2)

##### 1.2.3. Save as the **dim_employees** table in the Data Lakehouse

In [ ]:
df_dim_employees.write.saveAsTable(f"{dest_database}.dim_employees", mode="overwrite")

##### 1.2.4. Unit Test: Describe and Preview Table

In [ ]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_employees;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_employees LIMIT 5").toPandas()

### 2.0. Fetch Reference Data from MongoDB
#### 2.1. Load the Customers JSON file into a MongoDB Collection

In [ ]:
client = get_mongo_client(**mongodb_args)

json_files = {
    "customers": "dim_customers.json"
}

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files)
print("MongoDB collection loaded successfully.")

#### 2.2. Populate the **Customers Dimension**
##### 2.2.1. Fetch Data from the MongoDB Customers Collection

In [ ]:
mongodb_args["collection"] = "customers"

df_dim_customers = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_customers.toPandas().head(2)

##### 2.2.2. Make Necessary Transformations

In [ ]:
# Rename 'id' to 'customer_id'
df_dim_customers = df_dim_customers.withColumnRenamed("id", "customer_id")

# Add surrogate primary key
df_dim_customers.createOrReplaceTempView("customers")
sql_customers = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key
    FROM customers;
"""
df_dim_customers = spark.sql(sql_customers)

# Reorder columns – adjust to match the fields in your dim_customers.json
ordered_columns = ['customer_key', 'customer_id', 'first_name', 'last_name',
                   'email', 'phone', 'address', 'city', 'state', 'country']

existing_columns = [c for c in ordered_columns if c in df_dim_customers.columns]
df_dim_customers = df_dim_customers[existing_columns]
df_dim_customers.toPandas().head(2)

##### 2.2.3. Save as the **dim_customers** table in the Data Lakehouse

In [ ]:
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

##### 2.2.4. Unit Test: Describe and Preview Table

In [ ]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 5").toPandas()

### 3.0. Fetch Reference Data from MySQL
#### 3.1. Populate the **Date Dimension**
##### 3.1.1. Fetch data from the dim_date table in MySQL

In [ ]:
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)
df_dim_date.toPandas().head(2)

##### 3.1.2. Save as the **dim_date** table in the Data Lakehouse

In [ ]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

##### 3.1.3. Unit Test: Describe and Preview Table

In [ ]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 5").toPandas()

#### 3.2. Populate the **Product Dimension**
##### 3.2.1. Fetch data from the Products table in MySQL

In [ ]:
sql_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY ProductID) AS product_key
    FROM {mysql_args['db_name']}.product
"""
df_dim_products = get_mysql_dataframe(spark, sql_products, **mysql_args)
df_dim_products.toPandas().head(2)

##### 3.2.2. Make Necessary Transformations

In [ ]:
# Rename ProductID to product_id
df_dim_products = df_dim_products.withColumnRenamed("ProductID", "product_id")

# Drop any large text/blob columns not useful for analysis
for drop_col in ['description', 'LargePhoto', 'ThumbNailPhoto', 'ThumbnailPhotoFileName', 'rowguid', 'ModifiedDate']:
    if drop_col in df_dim_products.columns:
        df_dim_products = df_dim_products.drop(drop_col)

df_dim_products.toPandas().head(2)

##### 3.2.3. Save as the **dim_products** table in the Data Lakehouse

In [ ]:
df_dim_products.write.saveAsTable(f"{dest_database}.dim_products", mode="overwrite")

##### 3.2.4. Unit Test: Describe and Preview Table

In [ ]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_products;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_products LIMIT 5").toPandas()

### 4.0. Verify All Dimension Tables

In [ ]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

## Section III: Integrate Reference Data with Real-Time (Streaming) Sales Orders Data
### 5.0. Use PySpark Structured Streaming to Process Sales Orders Fact Data
#### 5.1. Verify the location of the source streaming files

In [ ]:
get_file_info(sales_orders_stream_dir)

#### 5.2. Create the Bronze Layer: Stage Raw Sales Orders Data
##### 5.2.1. Read Raw JSON files into a Stream

In [ ]:
df_sales_orders_bronze = (
    spark.readStream
    .option("schemaLocation", sales_orders_output_bronze)
    .option("maxFilesPerTrigger", 1)
    .option("multiLine", "true")
    .json(sales_orders_stream_dir)
)

df_sales_orders_bronze.isStreaming

##### 5.2.2. Write the Streaming Data to a Parquet file (Bronze)

In [ ]:
sales_orders_checkpoint_bronze = os.path.join(sales_orders_output_bronze, '_checkpoint')

sales_orders_bronze_query = (
    df_sales_orders_bronze
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    .writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("sales_orders_bronze")
    .trigger(availableNow=True)
    .option("checkpointLocation", sales_orders_checkpoint_bronze)
    .option("compression", "snappy")
    .start(sales_orders_output_bronze)
)

##### 5.2.3. Unit Test: Monitor the Bronze Query

In [ ]:
print(f"Query ID:     {sales_orders_bronze_query.id}")
print(f"Query Name:   {sales_orders_bronze_query.name}")
print(f"Query Status: {sales_orders_bronze_query.status}")

In [ ]:
sales_orders_bronze_query.awaitTermination()

#### 5.3. Create the Silver Layer: Integrate Cold-path Dimensions & Transform
##### 5.3.1. Prepare Role-Playing Date Dimension Views

In [ ]:
df_dim_order_date = df_dim_date.select(
    col("date_key").alias("order_date_key"),
    col("full_date").alias("order_full_date")
)

df_dim_ship_date = df_dim_date.select(
    col("date_key").alias("ship_date_key"),
    col("full_date").alias("ship_full_date")
)

df_dim_due_date = df_dim_date.select(
    col("date_key").alias("due_date_key"),
    col("full_date").alias("due_full_date")
)

##### 5.3.2. Define Silver Query – Join Streaming Fact Data with Dimension Tables

In [ ]:
df_sales_orders_silver = (
    spark.readStream.format("parquet").load(sales_orders_output_bronze)
    .join(df_dim_customers, "customer_id", "inner")
    .join(df_dim_employees, "employee_id", "left_outer")
    .join(df_dim_products,  "product_id",  "inner")
    .join(
        df_dim_order_date,
        df_dim_order_date.order_full_date.cast(DateType()) == col("order_date").cast(DateType()),
        "inner"
    )
    .join(
        df_dim_ship_date,
        df_dim_ship_date.ship_full_date.cast(DateType()) == col("ship_date").cast(DateType()),
        "left_outer"
    )
    .join(
        df_dim_due_date,
        df_dim_due_date.due_full_date.cast(DateType()) == col("due_date").cast(DateType()),
        "left_outer"
    )
    .select(
        col("sales_order_id").cast(LongType()),
        col("sales_order_detail_id").cast(LongType()),
        df_dim_customers.customer_key.cast(LongType()),
        df_dim_employees.employee_key.cast(LongType()),
        df_dim_products.product_key.cast(LongType()),
        df_dim_order_date.order_date_key.cast(LongType()),
        df_dim_ship_date.ship_date_key.cast(LongType()),
        df_dim_due_date.due_date_key.cast(LongType()),
        col("order_qty").cast(IntegerType()),
        col("unit_price").cast(DoubleType()),
        col("unit_price_discount").cast(DoubleType()),
        col("line_total").cast(DoubleType()),
        col("status"),
        col("online_order_flag")
    )
)

df_sales_orders_silver.isStreaming

In [ ]:
df_sales_orders_silver.printSchema()

##### 5.3.3. Write the Transformed Data to the Data Lakehouse (Silver)

In [ ]:
sales_orders_checkpoint_silver = os.path.join(sales_orders_output_silver, '_checkpoint')

sales_orders_silver_query = (
    df_sales_orders_silver.writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("sales_orders_silver")
    .trigger(availableNow=True)
    .option("checkpointLocation", sales_orders_checkpoint_silver)
    .option("compression", "snappy")
    .start(sales_orders_output_silver)
)

##### 5.3.4. Unit Test: Monitor the Silver Query

In [ ]:
print(f"Query ID:     {sales_orders_silver_query.id}")
print(f"Query Name:   {sales_orders_silver_query.name}")
print(f"Query Status: {sales_orders_silver_query.status}")

In [ ]:
sales_orders_silver_query.awaitTermination()

#### 5.4. Create the Gold Layer: Business-Level Aggregations
##### 5.4.1. Define a Business Report – Total Sales Revenue by Product Category per Month

In [ ]:
df_sales_by_category_gold = (
    spark.readStream.format("parquet").load(sales_orders_output_silver)
    .join(df_dim_products, "product_key")
    .join(
        df_dim_date,
        df_dim_date.date_key.cast(LongType()) == col("order_date_key").cast(LongType())
    )
    .groupBy("month_of_year", "month_name", "ProductCategory")
    .agg(
        count("product_key").alias("units_sold"),
        round(sum("line_total"), 2).alias("total_revenue")
    )
    .orderBy(asc("month_of_year"), desc("total_revenue"))
)

##### 5.4.2. Write Streaming Aggregation to Memory in 'Complete' Mode

In [ ]:
sales_gold_query = (
    df_sales_by_category_gold.writeStream
    .format("memory")
    .outputMode("complete")
    .queryName("fact_sales_by_category")
    .start()
)

wait_until_stream_is_ready(sales_gold_query, 1)

##### 5.4.3. Query the Gold Data from Memory

In [ ]:
df_fact_sales_by_category = spark.sql("SELECT * FROM fact_sales_by_category")
df_fact_sales_by_category.printSchema()

##### 5.4.4. Create the Final Selection

In [ ]:
df_fact_sales_by_category_final = (
    df_fact_sales_by_category
    .select(
        col("month_name").alias("Month"),
        col("ProductCategory").alias("Product Category"),
        col("units_sold").alias("Units Sold"),
        col("total_revenue").alias("Total Revenue ($)")
    )
    .orderBy(asc("month_of_year"), desc("Total Revenue ($)"))
)

##### 5.4.5. Save to the Data Lakehouse and Display Results

In [ ]:
df_fact_sales_by_category_final.write.saveAsTable(
    f"{dest_database}.fact_sales_by_product_category", mode="overwrite"
)
spark.sql(f"SELECT * FROM {dest_database}.fact_sales_by_product_category").toPandas()

### 6.0. Verify All Final Tables in the Data Lakehouse

In [ ]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

### 7.0. Stop the Spark Session

In [ ]:
spark.stop()